In [0]:
silver_df = spark.table("silver_customer_products")

schema = silver_df.schema

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("department_id", IntegerType()),
    StructField("aisle_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("add_to_cart_order", IntegerType()),
    StructField("reordered", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("eval_set", StringType()),
    StructField("order_number", IntegerType()),
    StructField("order_dow", IntegerType()),
    StructField("order_hour_of_day", IntegerType()),
    StructField("days_since_prior_order", DoubleType()),
    StructField("product_name", StringType()),
    StructField("aisle", StringType()),
    StructField("department", StringType())
])

In [0]:
stream_path = "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders"

stream_df = (
    spark.readStream
         .schema(schema)
         .option("header", True)
         .csv(stream_path)
)

In [0]:
from pyspark.sql.functions import count

trending = (
    stream_df
    .groupBy("product_id", "product_name")
    .agg(count("*").alias("purchases"))
)

In [0]:
query = (
    trending.writeStream
        .outputMode("complete")
        .format("memory")
        .queryName("trending_products")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

In [0]:
spark.sql("""
SELECT product_id,
       COUNT(DISTINCT product_name) AS names
FROM trending_products
GROUP BY product_id
HAVING names > 1
""").show(20, False)

+----------+-----+
|product_id|names|
+----------+-----+
+----------+-----+



In [0]:
spark.sql("""
SELECT COUNT(*)
FROM trending_products
""").show()

+--------+
|COUNT(*)|
+--------+
|   21580|
+--------+



In [0]:
spark.sql("""
SELECT *
FROM trending_products
ORDER BY purchases DESC
LIMIT 10
""").show(10,False)

+----------+----------------------+---------+
|product_id|product_name          |purchases|
+----------+----------------------+---------+
|24852     |Banana                |2993     |
|13176     |Bag of Organic Bananas|2390     |
|21137     |Organic Strawberries  |1573     |
|21903     |Organic Baby Spinach  |1480     |
|47209     |Organic Hass Avocado  |1334     |
|47766     |Organic Avocado       |1115     |
|47626     |Large Lemon           |927      |
|16797     |Strawberries          |906      |
|27966     |Organic Raspberries   |866      |
|26209     |Limes                 |861      |
+----------+----------------------+---------+



In [0]:
spark.sql("""
SELECT *
FROM trending_products
WHERE product_name='Banana'
""").show(20,False)

+----------+------------+---------+
|product_id|product_name|purchases|
+----------+------------+---------+
|24852     |Banana      |2993     |
+----------+------------+---------+



In [0]:
spark.sql("SHOW TABLES").show(100, False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|default |bronze_aisles           |false      |
|default |bronze_departments      |false      |
|default |bronze_orders           |false      |
|default |bronze_prior            |false      |
|default |bronze_products         |false      |
|default |bronze_train            |false      |
|default |clean_orders            |false      |
|default |gold_interaction_matrix |false      |
|default |silver_customer_products|false      |
|        |trending_products       |true       |
|        |trending_products_v2    |true       |
+--------+------------------------+-----------+



In [0]:
spark.sql("""

SELECT *
FROM trending_products
LIMIT 10

""").show(20,False)

+----------+----------------------------------------+---------+
|product_id|product_name                            |purchases|
+----------+----------------------------------------+---------+
|32771     |Organic Mango Acai Fruit Leather, 12 Ct |3        |
|27398     |Genuine Chocolate Flavor Syrup          |20       |
|25352     |Wasabi Powder                           |1        |
|9995      |Sesame Seed New York Style Bagels       |3        |
|48140     |All Natural Whole Wheat Bread           |6        |
|46863     |Organic Plain Coconut Milk Yogurt       |2        |
|26641     |Santa Maria Seasoning                   |2        |
|17426     |Simply Sea Salted Thick Cut Potato Chips|5        |
|49426     |Hommus Balsamic Caramelized Onion       |1        |
|26646     |Wintergreen Mints                       |2        |
+----------+----------------------------------------+---------+



In [0]:
top_products = spark.sql("""

SELECT *
FROM trending_products
ORDER BY purchases DESC
LIMIT 10

""")

display(top_products)

product_id,product_name,purchases
24852,Banana,2993
13176,Bag of Organic Bananas,2390
21137,Organic Strawberries,1573
21903,Organic Baby Spinach,1480
47209,Organic Hass Avocado,1334
47766,Organic Avocado,1115
47626,Large Lemon,927
16797,Strawberries,906
27966,Organic Raspberries,866
26209,Limes,861


In [0]:
trending_batch = (
    spark.table("trending_products")
)

trending_batch.write.mode("overwrite").saveAsTable(
    "top_trending_products"
)

In [0]:
spark.sql("SHOW TABLES").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|       bronze_aisles|      false|
| default|  bronze_departments|      false|
| default|       bronze_orders|      false|
| default|        bronze_prior|      false|
| default|     bronze_products|      false|
| default|        bronze_train|      false|
| default|        clean_orders|      false|
| default|gold_interaction_...|      false|
| default|silver_customer_p...|      false|
| default|top_trending_prod...|      false|
|        |   trending_products|       true|
|        |trending_products_v2|       true|
+--------+--------------------+-----------+

